# Stochastic Calculus, Step by Step · Lesson 23: The trinomial tree

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Why three branches, and why logs · 3. Matching the moments · 4. The stretch factor · 5. Pricing on the tree · 6. Against the binomial tree · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 23,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Why three branches, and why logs

Work with $X = \ln S$. Under the risk-neutral probability, $X$ moves with **constant** drift $\nu = r - \tfrac12\sigma^2$ and volatility $\sigma$ (Lesson 17). So one step has a mean and a variance that don't depend on where you are, and the same three moves, $+\Delta x$, $0$, $-\Delta x$, recombine everywhere.

In [ ]:
nu = r - 0.5 * sigma**2
n_steps = 100
dt = T / n_steps
out["setup"] = {"nu": nu, "n": n_steps, "dt": dt, "mean_step": nu * dt, "var_step": sigma**2 * dt, "sd_step": sigma * math.sqrt(dt)}
out["setup"]

## 3. Matching the moments

Take a jump size $\Delta x = \lambda\sigma\sqrt{\Delta t}$. Three unknown probabilities need three equations: they sum to 1, and they give the right mean and variance of $\Delta X$. Neglecting the very small $(\nu\Delta t)^2$:

$$p_u + p_m + p_d = 1, \qquad (p_u - p_d)\Delta x = \nu\Delta t, \qquad (p_u + p_d)\Delta x^2 = \sigma^2\Delta t .$$

The solution:

$$p_m = 1 - \frac{1}{\lambda^2}, \qquad p_{u,d} = \frac{1}{2\lambda^2} \pm \frac{\nu\sqrt{\Delta t}}{2\lambda\sigma} .$$

In [ ]:
def trin_probs(lam, n_steps_, sig=sigma, rate=r):
    dt_ = T / n_steps_
    dx_ = lam * sig * math.sqrt(dt_)
    nu_ = rate - 0.5 * sig**2
    pm_ = 1 - 1 / lam**2
    tilt = nu_ * dt_ / dx_
    return dx_, 0.5 * (1 / lam**2 + tilt), pm_, 0.5 * (1 / lam**2 - tilt)


lam3 = math.sqrt(3)
dx, pu, pm, pd_ = trin_probs(lam3, n_steps)
m1 = (pu - pd_) * dx; m2 = (pu + pd_) * dx**2; m3 = (pu - pd_) * dx**3; m4 = (pu + pd_) * dx**4
out["probs"] = {"lam": lam3, "dx": dx, "pu": pu, "pm": pm, "pd": pd_, "total": pu + pm + pd_, "m1": m1, "m1_target": nu * dt, "m2": m2,
                "m2_target": sigma**2 * dt, "m3": m3, "m4": m4, "m4_target": 3 * (sigma**2 * dt) ** 2,
                "growth": float(pu * math.exp(dx) + pm + pd_ * math.exp(-dx)), "growth_target": math.exp(r * dt)}
out["probs"]

## 4. The stretch factor

$\lambda$ is free. The fourth moment of one step is $(p_u + p_d)\Delta x^4 = \lambda^2\sigma^4\Delta t^2$, while a normal has $3\sigma^4\Delta t^2$. So $\lambda = \sqrt3$ gets the fourth moment right too, with the pleasing probabilities $\tfrac16, \tfrac23, \tfrac16$ before the small tilt. And $\lambda = 1$ throws the middle branch away: that is a binomial tree.

In [ ]:
lam_rows = []
for name, lam in (("1", 1.0), ("sqrt 2", math.sqrt(2)), ("sqrt 3", lam3), ("2", 2.0)):
    dx_, pu_, pm_, pd2 = trin_probs(lam, n_steps)
    lam_rows.append({"name": name, "lam": lam, "dx": dx_, "pu": pu_, "pm": pm_, "pd": pd2, "kurt": lam**2})
out["lam"] = {"rows": lam_rows}
# largest step at which p_d stays non-negative, with high and low rates
bound_rows = []
for rate in (0.03, 0.10, 0.25):
    nu_b = rate - 0.5 * sigma**2
    dt_max = sigma**2 / (lam3**2 * nu_b**2)
    bound_rows.append({"r_pct": rate * 100, "nu": nu_b, "dt_max": dt_max})
out["lam"]["bound"] = bound_rows
pd.DataFrame(lam_rows)

## 5. Pricing on the tree

Work backwards from the payoff: each node's value is the discounted average of its three children (Lesson 10's recipe, with one more branch). After $n$ steps there are $2n + 1$ nodes at the end.

In [ ]:
def trinomial(n_steps_, payoff, lam=lam3, american=False, sig=sigma, s0=S0):
    dx_, pu_, pm_, pd2 = trin_probs(lam, n_steps_, sig)
    df = math.exp(-r * T / n_steps_)
    idx = np.arange(-n_steps_, n_steps_ + 1)
    v = payoff(s0 * np.exp(idx * dx_))
    for i in range(n_steps_ - 1, -1, -1):
        v = df * (pu_ * v[2:] + pm_ * v[1:-1] + pd2 * v[:-2])
        if american:
            v = np.maximum(v, payoff(s0 * np.exp(np.arange(-i, i + 1) * dx_)))
    return float(v[0])


call_pay = lambda s: np.maximum(s - K, 0)
put_pay = lambda s: np.maximum(K - s, 0)
bs_c, bs_p = float(call(S0, T)), float(put(S0, T))
# two steps by hand, as a table of node values
dx2, pu2, pm2, pd22 = trin_probs(lam3, 2)
df2 = math.exp(-r * T / 2)
nodes_T = {str(j): {"S": S0 * math.exp(j * dx2), "V": max(S0 * math.exp(j * dx2) - K, 0)} for j in range(-2, 3)}
mid = {}
for j in (-1, 0, 1):
    mid[str(j)] = {"S": S0 * math.exp(j * dx2), "V": df2 * (pu2 * nodes_T[str(j + 1)]["V"] + pm2 * nodes_T[str(j)]["V"] + pd22 * nodes_T[str(j - 1)]["V"])}
root = df2 * (pu2 * mid["1"]["V"] + pm2 * mid["0"]["V"] + pd22 * mid["-1"]["V"])
out["small"] = {"dx": dx2, "pu": pu2, "pm": pm2, "pd": pd22, "df": df2, "end": nodes_T, "mid": mid, "root": root, "check": trinomial(2, call_pay)}
pr_rows = [{"n": n_, "call": trinomial(n_, call_pay), "err_call": trinomial(n_, call_pay) - bs_c, "put": trinomial(n_, put_pay),
            "err_put": trinomial(n_, put_pay) - bs_p, "nodes": (n_ + 1) ** 2} for n_ in (10, 50, 100, 500, 1000)]
out["price"] = {"rows": pr_rows, "bs_call": bs_c, "bs_put": bs_p}
pd.DataFrame(pr_rows)

## 6. Against the binomial tree

For the same number of steps a trinomial tree has about twice the nodes, so a fair comparison gives the binomial tree $\sqrt2$ times the steps. Judged by the worst error over ten neighbouring sizes, the two are close: the third branch alone is no great win, because the strike still falls in a different place among the final nodes each time.

The real win is that $\lambda$ is free. Choose the sizes so that the strike sits **exactly** on a node with $\lambda$ unchanged: $n = 50j^2$ for a whole number $j$ gives $j\,\Delta x = \ln(K/S_0)$ at $\lambda = 1.725$, very close to $\sqrt3$. The kink is now always in the same place relative to the grid. The error isn't smaller, but it is **steady**: exactly proportional to $1/n$. A steady error can be removed, by Richardson's trick (Lesson 22): since $n$ goes up by 4, take $(4V_{4n} - V_n)/3$.

In [ ]:
def binomial(n_steps_, payoff, american=False, s0=S0, sig=sigma):
    dt_ = T / n_steps_
    u_ = math.exp(sig * math.sqrt(dt_)); d_ = 1 / u_; p_ = (math.exp(r * dt_) - d_) / (u_ - d_); df = math.exp(-r * dt_)
    v = payoff(s0 * u_ ** (2 * np.arange(n_steps_ + 1) - n_steps_))
    for i in range(n_steps_ - 1, -1, -1):
        v = df * (p_ * v[1:] + (1 - p_) * v[:-1])
        if american:
            v = np.maximum(v, payoff(s0 * u_ ** (2 * np.arange(i + 1) - i)))
    return float(v[0])


def lam_on_strike(n_steps_):
    # The lambda nearest sqrt 3 that puts the strike exactly on a node.
    gap = math.log(K / S0)
    j = max(1, round(gap / (lam3 * sigma * math.sqrt(T / n_steps_))))
    return gap / (j * sigma * math.sqrt(T / n_steps_))


on_strike = lambda n_: trinomial(n_, call_pay, lam=lam_on_strike(n_))
cmp_rows = []
for n_ in (50, 100, 200, 400, 800):
    b_steps = round(math.sqrt(2) * n_)
    cmp_rows.append({"tri_steps": n_, "bin_steps": b_steps,
                     "bin_err": max(abs(binomial(b_steps + i, call_pay) - bs_c) for i in range(10)),
                     "tri_err": max(abs(trinomial(n_ + i, call_pay) - bs_c) for i in range(10))})
strike_rows = []
for j in (1, 2, 3, 4):
    n_ = 50 * j**2
    v1, v4 = on_strike(n_), on_strike(4 * n_)
    strike_rows.append({"j": j, "n": n_, "lam": lam_on_strike(n_), "err": v1 - bs_c, "err_n": (v1 - bs_c) * n_, "err4": v4 - bs_c, "rich": (4 * v4 - v1) / 3 - bs_c})
scan_n = list(range(40, 1000, 7))
out["cmp"] = {"rows": cmp_rows, "strike": strike_rows, "scan_n": scan_n, "scan_tri": [trinomial(n_, call_pay) - bs_c for n_ in scan_n],
              "points_n": [50 * j**2 for j in (1, 2, 3, 4)], "points_err": [x["err"] for x in strike_rows]}
pd.DataFrame(strike_rows)

In [ ]:
# Chart data for the video (no random numbers)
s_ = out["small"]
s_["nodes"] = [{"t": 0, "k": 0, "price": S0, "value": s_["root"]}]
s_["nodes"] += [{"t": 1, "k": int(j_), "price": v_["S"], "value": v_["V"]} for j_, v_ in s_["mid"].items()]
s_["nodes"] += [{"t": 2, "k": int(j_), "price": v_["S"], "value": v_["V"]} for j_, v_ in s_["end"].items()]
c_ = out["cmp"]
c_["scan_err_n"] = (np.array(c_["scan_tri"]) * np.array(c_["scan_n"])).tolist()
c_["points_err_n"] = [x_["err_n"] for x_ in c_["strike"]]
c_["ticks"] = [{"x": n_, "label": str(n_)} for n_ in (40, 200, 400, 600, 800, 1000)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Solve the three moment equations for $p_u$, $p_m$ and $p_d$ with $\Delta x = \lambda\sigma\sqrt{\Delta t}$, step by step. Evaluate them with $\lambda = \sqrt3$ and 100 steps.

**Exercise 2.** Compute the third moment $E[\Delta X^3]$ on the tree exactly. A normal step with mean $m = \nu\Delta t$ and variance $s^2 = \sigma^2\Delta t$ has $E[\Delta X^3] = m^3 + 3ms^2$. For which $\lambda$ do the two agree to leading order in $\Delta t$?

**Exercise 3.** Show that the tree's fourth moment is $\lambda^2\sigma^4\Delta t^2$ to leading order, and that the normal's is $3\sigma^4\Delta t^2$. For which $\lambda$ do they agree? Why does a binomial tree ($\lambda = 1$) get it wrong by a factor of 3?

**Exercise 4.** Find how small $\Delta t$ must be for $p_d \geq 0$ when $\lambda = \sqrt3$. What is the condition with the course's numbers? With $r = 25\%$? What goes wrong if a probability is negative?

**Exercise 5.** Price the call by hand on a one-step trinomial tree ($\Delta t = 1$, $\lambda = \sqrt3$), using $S_0 = 100$, and compare with 7.13. Why is one step so poor? Check also that the average discounted share on this tree is close to 100.

**Exercise 6.** Count the nodes in a trinomial tree of $n$ steps and in a binomial tree of $n$ steps. If one node's work is the same, how many binomial steps cost the same as $n$ trinomial steps? What is the total memory if you keep only one column?

**Exercise 7.** (Code) Build the trinomial pricer with $\lambda = \sqrt3$ and price the call and the put at $n = 10, 50, 100, 500$. Check put-call parity on the tree at each $n$: is it exact? Why, or why not?

**Exercise 8.** (Code) Compare $\lambda = 1.2, \sqrt2, \sqrt3, 2$ for the call by the worst error over $n = 200, \dots, 209$. Then price the American put at $n = 100, 200, 400, 800$ on the $\lambda = \sqrt3$ tree and compare with Lesson 22's value.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")